# Exploring Raw EHR Data for Structure and Quality Issues

Time estimate: **20** minutes

## Objectives
After completing this lab, you will be able to:
- Inspect the structure and schema of raw EHR data.  
- Identify missing values, duplicates, and inconsistencies.  
- Analyze temporal and clinical data quality issues.  
- Explain how data quality impacts downstream healthcare analytics.


## What you will do in this lab
In this lab, you will work with a synthetic EHR dataset to explore its structure and identify common data quality issues that can affect healthcare analysis.

You will:

- Load a synthetic EHR dataset.  
- Carefully explore how the data is organized.  
- Look for missing, duplicated, and inconsistent information.  
- Check whether dates and clinical values are reasonable.  
- Summarize key data quality issues found in the dataset.


## Overview
Electronic health record (EHR) data is created as part of routine clinical care and hospital operations.
Because it is collected for operational purposes rather than analysis, the data often contains gaps,
duplicates, and logical inconsistencies. Before any reporting, analytics, or machine learning can be
trusted, analysts must first explore the raw data and understand these issues. This lab focuses on
building that foundational inspection skill.


## About the dataset/environment
You will work with a **synthetic EHR dataset** designed to resemble real-world healthcare data.
The dataset includes patient identifiers, encounter identifiers, vital signs, and admission and
discharge dates. The data is intentionally imperfect to reflect what analysts typically encounter
when working with real hospital systems.


## Setup

In [2]:

# This cell imports required libraries and loads a synthetic EHR dataset.

import pandas as pd  # Used for working with tabular data
import numpy as np   # Used for generating random and numeric values

# Load a synthetic EHR dataset
ehr_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab1/ehr_dataset.csv")

# Display the first few rows to confirm dataset creation
ehr_df.head()


,patient_id,encounter_id,heart_rate,systolic_bp,diastolic_bp,admission_time,discharge_time
0,NaN,5075,85.0,140,90,2023-01-05,2023-01-16
1,NaN,5153,60.0,300,90,2023-01-15,2023-01-07
2,NaN,5143,60.0,120,80,2023-01-17,2023-01-23
3,NaN,5178,72.0,110,70,2023-01-03,2023-01-12
4,NaN,5085,72.0,140,70,2023-01-12,2023-01-23


## Step 1: Inspect dataset structure
In this step, you will begin by taking a broad look at the dataset. You will understand how many rows
and columns are present, what each column represents, and how the data is typed. This helps set
expectations before any deeper analysis.

**Why this matters in healthcare:** Misunderstanding data structure often leads to incorrect joins,
misinterpreted fields, and inaccurate reports.


In [3]:

# This cell displays the overall structure of the dataset.
# It helps understand the size of the data and how each column is stored.

# Display the number of rows and columns in the dataset
ehr_df.shape

# Display column names, data types, and non-null counts
ehr_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      489 non-null    float64
 1   encounter_id    500 non-null    int64  
 2   heart_rate      414 non-null    float64
 3   systolic_bp     500 non-null    int64  
 4   diastolic_bp    500 non-null    int64  
 5   admission_time  500 non-null    str    
 6   discharge_time  500 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 27.5 KB


## Step 2: Evaluate identifier quality
Here, you will focus on patient and encounter identifiers. These fields are used to link records across
tables and time. You will check whether these identifiers are missing and whether they behave as expected.

**Why this matters in healthcare:** Missing or inconsistent identifiers can fragment patient records
or incorrectly combine data from different individuals.


In [4]:

# This cell checks whether identifier fields are complete and consistent.
# Reliable identifiers are critical for accurate patient-level analysis.

# Count missing values in identifier columns
ehr_df[['patient_id', 'encounter_id']].isnull().sum()

# Count how many unique patients and encounters exist
ehr_df['patient_id'].nunique()
ehr_df['encounter_id'].nunique()


183

## Step 3: Analyze missing data
In this step, you will examine how much data is missing and where those gaps occur. Some missing values
might be expected, while others can signal data capture or system issues.

**Why this matters in healthcare:** Missing clinical or demographic data can introduce bias and
reduce confidence in analytical results.


In [5]:

# This cell calculates how many values are missing in each column.
# Understanding missingness helps assess data completeness.

# Count missing values per column
ehr_df.isnull().sum()


patient_id        11
encounter_id       0
heart_rate        86
systolic_bp        0
diastolic_bp       0
admission_time     0
discharge_time     0
dtype: int64

## Step 4: Detect duplicate records
Next, you will check whether the same record appears more than once in the dataset. Duplicate rows can
occur due to system errors or data integration issues.

**Why this matters in healthcare:** Duplicate records can inflate patient counts, visit volumes,
and outcome measures.


In [6]:

# This cell identifies duplicate rows in the dataset.
# Duplicate records can distort summary statistics and trends.

# Count the number of duplicate rows
ehr_df.duplicated().sum()


np.int64(0)

## Step 5: Validate temporal consistency
Here, you will verify that date fields follow a logical order. For example, a patient should not be
discharged before they are admitted.

**Why this matters in healthcare:** Incorrect timelines lead to wrong length-of-stay calculations
and misleading operational metrics.


In [7]:

# This cell checks for illogical date sequences.
# Time-related errors can lead to incorrect duration calculations.

# Find records where discharge occurs before admission
ehr_df[ehr_df['discharge_time'] < ehr_df['admission_time']].head()


,patient_id,encounter_id,heart_rate,systolic_bp,diastolic_bp,admission_time,discharge_time
1,NaN,5153,60.0,300,90,2023-01-15,2023-01-07
7,NaN,5068,110.0,120,90,2023-01-23,2023-01-18
8,NaN,5046,NaN,110,90,2023-01-21,2023-01-19
10,NaN,5189,110.0,300,80,2023-01-22,2023-01-19
15,1052.0,5038,85.0,300,80,2023-01-28,2023-01-16


## Step 6: Inspect clinical value ranges
Finally, you will review whether clinical measurements fall within reasonable ranges. Extremely high or
low values might indicate data entry mistakes or system issues.

**Why this matters in healthcare:** Implausible clinical values can mislead analyses and compromise
patient safety insights.


In [8]:

# This cell summarizes clinical measurements.
# Extreme or unrealistic values might signal data quality problems.

# Generate summary statistics for vital signs
ehr_df[['heart_rate', 'systolic_bp', 'diastolic_bp']].describe()


,heart_rate,systolic_bp,diastolic_bp
count,414.000000,500.000000,500.000000
mean,108.403382,163.980000,79.700000
std,52.040626,75.236165,8.014891
min,60.000000,110.000000,70.000000
25%,72.000000,110.000000,70.000000
50%,85.000000,120.000000,80.000000
75%,110.000000,140.000000,90.000000
max,200.000000,300.000000,90.000000


## Exercises

In [9]:
# Load the dataset for exercises
# Load a synthetic EHR dataset
ehr_exercise_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab1/ehr_exercise_dataset.csv")
ehr_exercise_df.head()

,patient_id,encounter_id,heart_rate,systolic_bp,diastolic_bp,admission_time,discharge_time
0,1534.0,6081,88.0,145,92,2023-02-03,2023-02-16
1,1503.0,6057,75.0,145,92,2023-02-24,2023-02-05
2,1564.0,6163,NaN,125,82,2023-02-08,2023-02-01
3,1587.0,6157,75.0,125,82,2023-02-11,2023-02-06
4,1517.0,6054,75.0,115,92,2023-02-19,2023-02-17


### Exercise 1: Inspect dataset structure

In [12]:
# your code goes here

ehr_exercise_df.shape
ehr_exercise_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      489 non-null    float64
 1   encounter_id    500 non-null    int64  
 2   heart_rate      426 non-null    float64
 3   systolic_bp     500 non-null    int64  
 4   diastolic_bp    500 non-null    int64  
 5   admission_time  500 non-null    str    
 6   discharge_time  500 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 27.5 KB


<details>
<summary>Click here for a hint</summary>

Look at dataset size and data types.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df.shape
ehr_exercise_df.info()
```

</details>


### Exercise 2: Check identifier completeness

In [13]:
# your code goes here

ehr_exercise_df[['patient_id', 'encounter_id']].isnull().sum()

patient_id      11
encounter_id     0
dtype: int64

<details>
<summary>Click here for a hint</summary>

Check for missing patient or encounter IDs.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df[['patient_id','encounter_id']].isnull().sum()
```

</details>


### Exercise 3: Identify missing data patterns

In [14]:
# your code goes here
ehr_exercise_df.isnull().sum()

patient_id        11
encounter_id       0
heart_rate        74
systolic_bp        0
diastolic_bp       0
admission_time     0
discharge_time     0
dtype: int64

<details>
<summary>Click here for a hint</summary>

Count missing values per column.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df.isnull().sum()
```

</details>


### Exercise 4: Detect duplicate records

In [15]:
# your code goes here

ehr_exercise_df.duplicated().sum()

np.int64(0)

<details>
<summary>Click here for a hint</summary>

Check whether rows are repeated.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df.duplicated().sum()
```

</details>


### Exercise 5: Identify invalid timestamps

In [16]:
# your code goes here
ehr_exercise_df[ehr_exercise_df['discharge_time'] < ehr_exercise_df['admission_time']]

,patient_id,encounter_id,heart_rate,systolic_bp,diastolic_bp,admission_time,discharge_time
1,1503.0,6057,75.0,145,92,2023-02-24,2023-02-05
2,1564.0,6163,NaN,125,82,2023-02-08,2023-02-01
3,1587.0,6157,75.0,125,82,2023-02-11,2023-02-06
4,1517.0,6054,75.0,115,92,2023-02-19,2023-02-17
6,NaN,6081,88.0,280,82,2023-02-06,2023-02-01
...,...,...,...,...,...,...,...
490,1554.0,6153,88.0,280,92,2023-02-10,2023-02-01
493,1517.0,6151,62.0,125,92,2023-02-21,2023-02-10
495,1581.0,6195,NaN,145,92,2023-02-27,2023-02-10
497,1578.0,6147,88.0,115,92,2023-02-27,2023-02-15


<details>
<summary>Click here for a hint</summary>

Compare admission and discharge dates.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df[ehr_exercise_df['discharge_time'] < ehr_exercise_df['admission_time']]
```

</details>


### Exercise 6: Review clinical value ranges

In [17]:
# your code goes here

ehr_exercise_df[['heart_rate', 'systolic_bp', 'diastolic_bp']].describe()

,heart_rate,systolic_bp,diastolic_bp
count,426.000000,500.000000,500.000000
mean,103.422535,167.120000,82.100000
std,45.208483,66.504521,8.069711
min,62.000000,115.000000,72.000000
25%,75.000000,125.000000,72.000000
50%,88.000000,145.000000,82.000000
75%,105.000000,280.000000,92.000000
max,190.000000,280.000000,92.000000


<details>
<summary>Click here for a hint</summary>

Summarize vital sign distributions.

</details>

<details>
<summary>Click here for solution</summary>

```python
ehr_exercise_df[['heart_rate','systolic_bp','diastolic_bp']].describe()
```

</details>
